<a href="https://colab.research.google.com/github/shahabday/DSR-practical-computer-vision/blob/main/Mlops_oxford_pet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!nvidia-smi




/bin/bash: line 1: nvidia-smi: command not found


In [4]:
import wandb
import os
from google.colab import userdata


api_key = userdata.get('wandb_api')

os.environ
os.environ['WANDB_API_key'] = api_key

wandb.login()

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: shahabdaiani (shahabdaiani-dsr). Use `wandb login --relogin` to force relogin


True

In [5]:
import os
import io
import random
import numpy as np
import pandas as pd
import skimage.io as io
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.notebook import tqdm
from google.colab import userdata
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import OneCycleLR
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2 as transforms
from torchvision import models
from torchvision.datasets import OxfordIIITPet
from sklearn.metrics import confusion_matrix, classification_report, precision_score, recall_score, accuracy_score
from tqdm import tqdm
from transformers import CLIPModel, CLIPProcessor


# Initialize Wandb

In [8]:
#TODO: Actually use these values
config = {
               "architecture": "resnet18",
               "dataset": "Oxford-IIIT Pet Dataset", # TODO: reformat this
               "learning_rate": 0.001,
               "epochs": 10,
               "batch_size": 32,
               "optimizer": "Adam",
               "scheduler": "OneCycleLR",
               "loss_function": "CrossEntropyLoss"
           }

In [10]:
# We use this to start wandb and specify the project that logs our experiments
wandb.init(project='dsr-pets-breed-classification-oxford-dataset',
           # Folder inside the project
           name='supervised-learning',
           # config has the hyperparameters that give a brief overview of what we are doing
           config=config)

# Define data transformation


In [13]:
# Data Preparation
# Define transformations applied to all sets during training and evaluation with DataLoaders
# These transformations are to be applied 'on the fly', ideally these would be persisted on disk
# before training and evaluation, so that we can carefully explore the datasets, their relationships with one another
# and the model's performamce.

# We apply Imagenet normalization on all sets (train, test, and val)

# We do data augmentation in the training set only
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    # Random transformations should only be applied on the training set
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    #
    transforms.ToImage(),
    # ToDtype turns the uint8 tensors to torch.float32 range
    # and scales them in the range [0.0, 1.0] by dividing all intensities with 255.
    transforms.ToDtype(torch.float32, scale=True),
    # This applies the standard scaler considering the Imagenet stats for R, G, B intensities
    # We need to do this because we are doing transfer learning, we are going to be using
    # an Imagenet pretrained model and we benefit from making our R, G, B intensities be in the same scale
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# The transformations that we apply in the validation and test sets
# are meant to keep evaluations of cross entropy loss and performance metrics consisten
val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToImage(),
    transforms.ToDtype(torch.float32, scale=True),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:

# Load the dataset
print("Loading Oxford-IIIT Pet Dataset...")
data_root = './data'
os.makedirs(data_root, exist_ok=True)

# Download the dataset
train_dataset = OxfordIIITPet(root=data_root, split='trainval', download=True, transform=train_transforms)
test_dataset = OxfordIIITPet(root=data_root, split='test', download=True, transform=val_test_transforms)

# Check sizes
len(train_dataset), len(test_dataset)